# 01 · State, nodes and edges

After this notebook you can build, run, draw and stream a LangGraph `StateGraph`, and explain in an interview why agent systems use a graph instead of a chain or a hand-written `while` loop.

**Time:** ~25 min · **Runs:** offline on CPU in < 30 s, no LLM needed · **Needs:** basic Python; [LCEL runnables](../13_langchain/02_lcel_runnables_and_output_parsers.ipynb) helps but is optional

## Why this matters in interviews

- "LangChain vs LangGraph?" is a warm-up question in most agent interviews. The crisp answer is *components vs orchestration*, and you should be able to sketch a `StateGraph` on a whiteboard.
- Interviewers probe the state model: why nodes return partial updates, what happens when two nodes write the same key, and why `add_messages` exists.
- "Agent vs workflow" questions are really about **control flow**. A graph makes control flow explicit, testable and resumable, which is the argument you want to be able to make.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `tl02` | LangChain, LangGraph and LlamaIndex — what is the difference? |
| `ag01` | What is an AI agent, and how is it different from a chain? |
| `ag29` | What is the difference between an agent and a workflow, in production terms? |
| `fo32` | What is the difference between an LLM, an agent and a workflow? |
| `ag40` | What is the difference between an agent framework and rolling your own? |
| `tl17` | Compare CrewAI, AutoGen and LangGraph. |

## 1. Why a graph?

**In plain English:** a chain is a recipe with fixed steps. An agent also has to *go back* ("that failed, try again"), *branch* ("is this billing or technical?"), *pause* ("wait for a human to approve the refund") and *survive a restart*. A graph is a flowchart that the program actually runs: **nodes** (boxes) do the work, **edges** (arrows) say what runs next, and a shared **state** (a dict) carries the data between boxes.

| You need... | LCEL chain | Hand-written `while` loop | LangGraph |
|---|---|---|---|
| A fixed sequence of steps | yes | yes | yes |
| Branches | awkward (`RunnableBranch`) | `if/else` | conditional edges (notebook 02) |
| Loops / retries | no | yes | yes, with a `recursion_limit` safety net |
| Shared, typed state | pass dicts along | your own variables | a state schema + reducers (this notebook) |
| Pause for a human, resume later | no | you build it | `interrupt()` + a checkpointer (notebook 04) |
| Save every step, rewind, fork | no | you build it | checkpointer + time travel (notebook 03) |
| Stream progress step by step | partly | you build it | `stream(stream_mode=...)` (this notebook) |
| A diagram of the control flow | partly | no | `get_graph()` (this notebook) |

The whole mental model is three words:

```
          STATE  -- one dict every node reads; the only thing that flows between nodes
          ┌──────────────────────────────────────────────┐
          │ ticket   clean   category   reply            │
          └──────────────────────────────────────────────┘
 START ──► normalise ──► categorise ──► draft_reply ──► END      <- EDGES say what runs next
           returns       returns        returns                  <- NODES are plain functions
           {"clean"}     {"category"}   {"reply"}                   that return only what changed
```

You declare the state schema, add nodes and edges to a `StateGraph` builder, then `compile()` it into a runnable object with `invoke`, `stream` and `batch`.

## 2. Setup

No model and no API key in this notebook: every node is ordinary Python, so every output is deterministic. The LLM arrives in notebook 03.

In [ ]:
import operator
from importlib.metadata import version
from typing import Annotated, TypedDict

import matplotlib.pyplot as plt
from langgraph.graph import StateGraph, START, END

for pkg in ["langgraph", "langgraph-checkpoint", "langchain-core"]:
    print(f"{pkg:22s} {version(pkg)}")
print("START is", repr(START), "| END is", repr(END))

A small helper that draws any compiled graph with matplotlib (the built-in PNG renderers need an internet service or Graphviz). You do not need to read it; it is reused in every notebook of this section.

In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

def draw_graph(app, title=None, xray=False):
    """Draw a compiled LangGraph left to right: solid = fixed edge, dashed = conditional edge,
    arcs above = skip ahead, arcs below = loop back. Red text = the path-map label."""
    g = app.get_graph(xray=xray)
    edges = [(e.source, e.target, e.conditional, e.data) for e in g.edges]
    col, frontier = {"__start__": 0}, ["__start__"]
    while frontier:                                     # column = hops from START (BFS)
        nxt = []
        for n in frontier:
            for s, t, _, _ in edges:
                if s == n and t not in col and t != "__end__":
                    col[t] = col[n] + 1; nxt.append(t)
        frontier = nxt
    for n in g.nodes:
        col.setdefault(n, max(col.values()) + 1)
    col["__end__"] = max(c for n, c in col.items() if n != "__end__") + 1
    rows = {}
    for n in g.nodes:
        rows.setdefault(col[n], []).append(n)
    pos = {n: (c * 2.4, (len(ns) - 1) / 2 - i) for c, ns in rows.items() for i, n in enumerate(ns)}
    nrow = max(len(v) for v in rows.values())
    fig, ax = plt.subplots(figsize=(min(11, 2.0 * len(rows) + 0.6), min(4.5, 0.9 * nrow + 1.4)))
    boxes = {}
    for n, (x, y) in pos.items():
        parts = [{"__start__": "START", "__end__": "END"}.get(p, p) for p in n.split(":")]
        fc = {"START": "#d3f9d8", "END": "#ffe3e3"}.get(parts[-1], "#e7f0ff")
        boxes[n] = FancyBboxPatch((x - 0.8, y - 0.28), 1.6, 0.56, boxstyle="round,pad=0.05", fc=fc, ec="#3b5bdb")
        ax.add_patch(boxes[n])
        ax.text(x, y, "\n".join(parts), ha="center", va="center", fontsize=8.5 if len(parts) == 1 else 7.5)
    pairs = {(s, t) for s, t, _, _ in edges}
    for s, t, cond, data in edges:
        (x1, y1), (x2, y2) = pos[s], pos[t]
        span = col[t] - col[s]
        if (t, s) in pairs and abs(span) == 1:
            rad = 0.2                                   # two-way pair: split the arrows apart
        elif span == 1 or (span == 0 and s != t):
            rad = 0.0
        else:
            rad = -0.3 if span > 1 else -0.45           # skip ahead above, loop back below
        ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), patchA=boxes[s], patchB=boxes[t], arrowstyle="-|>",
                                     mutation_scale=12, lw=1.3, color="#495057", linestyle="--" if cond else "-",
                                     connectionstyle=f"arc3,rad={rad}"))
        if data is not None and str(data) != t:
            mx, my = (x1 + x2) / 2 + 0.5 * rad * (y2 - y1), (y1 + y2) / 2 - 0.5 * rad * (x2 - x1)
            ax.text(mx, my + 0.1, str(data), ha="center", fontsize=7.5, color="#c92a2a")
    xs = [p[0] for p in pos.values()]; ys = [p[1] for p in pos.values()]
    ax.set_xlim(min(xs) - 1.0, max(xs) + 1.0); ax.set_ylim(min(ys) - 0.75, max(ys) + 0.85)
    ax.axis("off"); ax.set_title(title or "graph", fontsize=10)
    plt.tight_layout(); plt.show()

## 3. Your first graph: a support-ticket pipeline

Three steps, all plain functions. A node receives the **whole current state** and returns a dict with **only the keys it changed**.

In [ ]:
class TicketState(TypedDict):
    ticket: str      # raw text from the customer
    clean: str       # normalised text
    category: str    # billing | technical | general
    reply: str       # the draft answer

def normalise(state: TicketState) -> dict:
    return {"clean": " ".join(state["ticket"].lower().split())}

def categorise(state: TicketState) -> dict:
    text = state["clean"]
    if any(w in text for w in ("refund", "invoice", "charged")):
        return {"category": "billing"}
    if any(w in text for w in ("error", "crash", "login")):
        return {"category": "technical"}
    return {"category": "general"}

REPLIES = {"billing": "Sorry about the billing problem - I have passed it to our payments team.",
           "technical": "Thanks for the report - an engineer will look at the error today.",
           "general": "Thanks for getting in touch - we will reply within one working day."}

def draft_reply(state: TicketState) -> dict:
    return {"reply": REPLIES[state["category"]]}

Now wire them up. `START` and `END` are virtual nodes that mark where a run enters and leaves.

In [ ]:
builder = StateGraph(TicketState)
builder.add_node("normalise", normalise)
builder.add_node("categorise", categorise)
builder.add_node("draft_reply", draft_reply)
builder.add_edge(START, "normalise")
builder.add_edge("normalise", "categorise")
builder.add_edge("categorise", "draft_reply")
builder.add_edge("draft_reply", END)

pipeline = builder.compile()          # validates the wiring, returns a runnable
print("compiled type:", type(pipeline).__name__)

result = pipeline.invoke({"ticket": "  I was CHARGED twice,   please refund me  "})
for key, value in result.items():
    print(f"{key:9s} -> {value!r}")
assert result["category"] == "billing" and result["clean"].startswith("i was charged twice")

We passed in only `ticket`. Each node filled in one more key, and `invoke` returned the final state. Note that `ticket` still holds the original text: nobody returned it, so nobody overwrote it.

`compile()` is also where wiring mistakes are caught, before anything runs:

In [ ]:
bad = StateGraph(TicketState)
bad.add_node("normalise", normalise)
bad.add_edge(START, "normalise")
bad.add_edge("normalise", "categorize")        # typo: that node does not exist
try:
    bad.compile()
except ValueError as e:
    print("compile error 1:", e)

no_entry = StateGraph(TicketState)
no_entry.add_node("normalise", normalise)       # a node, but no edge from START
try:
    no_entry.compile()
except ValueError as e:
    print("compile error 2:", e)

## 4. State updates are merged, not replaced

**In plain English:** every node hands back a sticky note with just what it changed, and LangGraph sticks it onto the shared state. That is why the nodes above could return one key each.

Three gotchas follow directly from that design, and interviewers like all three:

In [ ]:
class Tiny(TypedDict):
    text: str
    note: str

def run_single(node_fn, start):
    g = StateGraph(Tiny)
    g.add_node("n", node_fn)
    g.add_edge(START, "n"); g.add_edge("n", END)
    return g.compile().invoke(start)

def mutate_in_place(state):          # gotcha 1: editing the dict you were given
    state["text"] = "CHANGED"
    return {}

def misspelt_key(state):             # gotcha 2: a key that is not in the schema
    return {"nte": "typo"}

def not_a_dict(state):               # gotcha 3: forgetting to wrap the update in a dict
    return "hello"

start = {"text": "original", "note": ""}
r1 = run_single(mutate_in_place, start)
r2 = run_single(misspelt_key, start)
print("1. in-place mutation ->", r1)
print("2. misspelt key      ->", r2)
try:
    run_single(not_a_dict, start)
except Exception as e:
    print("3. non-dict return   ->", type(e).__name__, "|", str(e).splitlines()[0])
assert r1["text"] == "original"        # the mutation was NOT persisted
assert "nte" not in r2 and r2["note"] == ""   # the typo vanished silently

1. **Mutating the state in place does nothing.** The node gets a copy; only the returned dict counts.
2. **A misspelt key is silently dropped.** No error, the update just disappears. Unit-test your nodes.
3. **A non-dict return is an error** (`InvalidUpdateError`). Returning `{}` or `None` means "no change".

## 5. Reducers: how one key combines several updates

By default a key is **overwritten** by the latest update ("last write wins"). A **reducer** changes that. You attach it with `Annotated[type, reducer]`, where the reducer is any function `(current_value, update) -> new_value`.

The most common one is `operator.add`, which turns a list key into an append-only log:

In [ ]:
class AuditedTicket(TicketState):              # same keys as before, plus a log
    audit: Annotated[list, operator.add]

def logged(name, fn):
    """Wrap a node so it also appends its name to the audit log."""
    return lambda state: {**fn(state), "audit": [name]}

b = StateGraph(AuditedTicket)
for name, fn in [("normalise", normalise), ("categorise", categorise), ("draft_reply", draft_reply)]:
    b.add_node(name, logged(name, fn))
b.add_edge(START, "normalise"); b.add_edge("normalise", "categorise")
b.add_edge("categorise", "draft_reply"); b.add_edge("draft_reply", END)
audited = b.compile()

out = audited.invoke({"ticket": "The app shows an error at login", "audit": ["received"]})
print("category:", out["category"])
print("audit   :", out["audit"])
assert out["audit"] == ["received", "normalise", "categorise", "draft_reply"]

Each node returned a one-item list and the reducer **appended** it instead of overwriting.

Reducers matter most when branches run **in parallel**. LangGraph executes in *super-steps*: every node triggered in the same step runs concurrently on the same input state, and their updates are applied together at the end of the step. Two branches writing one plain key in the same step is ambiguous, so LangGraph refuses:

In [ ]:
def parallel_graph(state_cls):
    g = StateGraph(state_cls)
    g.add_node("keyword_check", lambda s: {"flags": ["keyword: refund"]})
    g.add_node("sentiment_check", lambda s: {"flags": ["sentiment: angry"]})
    g.add_edge(START, "keyword_check"); g.add_edge(START, "sentiment_check")   # fan out: both run in step 1
    g.add_edge("keyword_check", END); g.add_edge("sentiment_check", END)
    return g.compile()

class PlainFlags(TypedDict):
    flags: list                                   # no reducer: last write wins... but which one?

class ReducedFlags(TypedDict):
    flags: Annotated[list, operator.add]          # reducer: keep both

try:
    parallel_graph(PlainFlags).invoke({"flags": []})
except Exception as e:
    print("without reducer ->", type(e).__name__, "|", str(e).splitlines()[0])

both = parallel_graph(ReducedFlags).invoke({"flags": []})
print("with reducer    ->", both["flags"])
assert sorted(both["flags"]) == ["keyword: refund", "sentiment: angry"]

A custom reducer is just a function. Here is a dict-merging one, the shape you will use for a shared "notes" blackboard in the multi-agent notebook:

In [ ]:
def merge_dicts(current: dict, update: dict) -> dict:
    return {**(current or {}), **update}

class Notes(TypedDict):
    notes: Annotated[dict, merge_dicts]

g = StateGraph(Notes)
g.add_node("research", lambda s: {"notes": {"price": "$30 per seat"}})
g.add_node("policy", lambda s: {"notes": {"refunds": "within 14 days"}})
g.add_edge(START, "research"); g.add_edge(START, "policy")
g.add_edge("research", END); g.add_edge("policy", END)
notes = g.compile().invoke({"notes": {"customer": "ACME"}})["notes"]
print(notes)
assert set(notes) == {"customer", "price", "refunds"}

| Reducer | What an update does | Typical key |
|---|---|---|
| none (default) | overwrites the value | `category`, `status`, `draft` |
| `operator.add` | concatenates lists (or adds numbers) | audit log, parallel results, counters |
| `add_messages` | appends messages, replaces by `id`, deletes with `RemoveMessage` | `messages` in any chat agent |
| your own function | anything, e.g. merge dicts, keep the max | shared notes, best score so far |

## 6. Messages: `add_messages` and `MessagesState`

**In plain English:** a chat history is a list, but agents also need to *edit* a message (a human corrects a tool call before it runs) and *delete* old ones (trim the context window). `operator.add` can only append. `add_messages` appends new messages, **replaces** a message whose `id` already exists, and **removes** a message when it receives `RemoveMessage(id=...)`.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, RemoveMessage
from langgraph.graph import add_messages, MessagesState

def show(label, msgs):
    print(f"{label:22s}", [(m.id, m.type, m.content) for m in msgs])

history = add_messages([], [HumanMessage("hi", id="m1"), AIMessage("hello!", id="m2")])
show("append two:", history)
history = add_messages(history, [AIMessage("Hello! How can I help?", id="m2")])
show("same id -> replace:", history)
history = add_messages(history, [RemoveMessage(id="m1")])
show("RemoveMessage:", history)

naive = operator.add([AIMessage("hello!", id="m2")], [AIMessage("Hello! How can I help?", id="m2")])
print("operator.add instead  ", [(m.id, m.content) for m in naive], "<- duplicated, not replaced")
assert len(history) == 1 and history[0].content == "Hello! How can I help?"
assert len(naive) == 2

`MessagesState` is a ready-made schema with exactly one key, `messages: Annotated[list[AnyMessage], add_messages]`. Subclass it to add your own keys. It also accepts plain `("user", "...")` tuples or `{"role": ..., "content": ...}` dicts and converts them to message objects (with generated ids, which is why we don't print them here).

In [ ]:
class ChatState(MessagesState):
    turns: Annotated[int, operator.add]           # a counter: each update adds to it

def echo_bot(state: ChatState) -> dict:
    last = state["messages"][-1]
    return {"messages": [AIMessage(f"You said: {last.content}")], "turns": 1}

g = StateGraph(ChatState)
g.add_node("bot", echo_bot)
g.add_edge(START, "bot"); g.add_edge("bot", END)
chat = g.compile()

out = chat.invoke({"messages": [("user", "is this thing on?")], "turns": 0})
for m in out["messages"]:
    print(f"{type(m).__name__:12s} {m.content}")
print("turns:", out["turns"])
assert [m.type for m in out["messages"]] == ["human", "ai"] and out["turns"] == 1

## 7. A shortcut for straight pipelines: `add_sequence`

For a linear pipeline, `add_sequence` adds the nodes (named after the functions) and the edges between them. You still connect `START`.

In [ ]:
seq = StateGraph(TicketState)
seq.add_sequence([normalise, categorise, draft_reply])
seq.add_edge(START, "normalise")
seq_app = seq.compile()

ticket = {"ticket": "Where is my parcel?"}
assert seq_app.invoke(ticket) == pipeline.invoke(ticket)
print("add_sequence gives the same result:", seq_app.invoke(ticket)["category"])

## 8. Drawing the graph

`get_graph()` returns the graph's structure. Three ways to look at it:

- `draw_ascii()`: text art in the notebook (uses the `grandalf` package). Fine for straight lines, but it can hide loop-back edges, as notebook 02 shows.
- `draw_mermaid()`: Mermaid source text. Paste it into [mermaid.live](https://mermaid.live) or a Markdown file on GitHub to get a diagram. (`draw_mermaid_png()` exists but calls the mermaid.ink web service by default, so it is not used offline.)
- `nodes` / `edges`: plain data, so a **unit test can assert the wiring** of your agent.

In [ ]:
print(pipeline.get_graph().draw_ascii())

In [ ]:
print(pipeline.get_graph().draw_mermaid())

In [ ]:
edges = {(e.source, e.target) for e in pipeline.get_graph().edges}
print(sorted(edges))
assert ("categorise", "draft_reply") in edges and ("draft_reply", "__end__") in edges

draw_graph(pipeline, "Support-ticket pipeline: four fixed edges, no branches")

## 9. Streaming each step

`invoke` gives you the final state. `stream` yields events **as each step finishes**, which is what a UI needs ("Categorising...", "Drafting..."). The `stream_mode` decides what each event contains:

| `stream_mode` | Each event is | Use it for |
|---|---|---|
| `"values"` | the full state after each step (the first event is the input) | debugging, showing the whole picture |
| `"updates"` | `{node_name: that node's partial update}` | progress UIs, logging who changed what |
| `"messages"` | LLM tokens as they are generated, plus metadata | chat UIs (notebook 05) |
| `"custom"` | anything a node sends with `get_stream_writer()` | progress bars from inside long nodes |
| `"debug"` / `"tasks"` / `"checkpoints"` | detailed task and checkpoint events | deep debugging |

Pass a list of modes to get `(mode, payload)` tuples.

In [ ]:
ticket = {"ticket": "Refund please, I was charged twice"}

print('stream_mode="values"')
for i, state in enumerate(pipeline.stream(ticket, stream_mode="values")):
    print(f"  event {i}: keys so far = {sorted(state)}")

print('stream_mode="updates"')
for update in pipeline.stream(ticket, stream_mode="updates"):
    for node, delta in update.items():
        print(f"  {node:12s} wrote {delta}")

In [ ]:
from langgraph.config import get_stream_writer

def slow_step(state: TicketState) -> dict:
    write = get_stream_writer()
    for pct in (50, 100):
        write({"progress": f"categorising {pct}%"})   # goes to stream_mode="custom"
    return categorise(state)

sb = StateGraph(TicketState)
sb.add_node("normalise", normalise); sb.add_node("categorise", slow_step)
sb.add_edge(START, "normalise"); sb.add_edge("normalise", "categorise"); sb.add_edge("categorise", END)

events = list(sb.compile().stream(ticket, stream_mode=["updates", "custom"]))
for mode, payload in events:
    print(f"{mode:8s} {payload}")
assert [m for m, _ in events].count("custom") == 2

The same `updates` stream, drawn as a grid: one row per step, one column per state key. It makes the "partial update" idea visible, because each node touches exactly one column.

In [ ]:
keys = list(TicketState.__annotations__)
steps, grid = [], []
for update in pipeline.stream(ticket, stream_mode="updates"):
    for node, delta in update.items():
        steps.append(node)
        grid.append([1 if k in delta else 0 for k in keys])

fig, ax = plt.subplots(figsize=(7, 2.8))
ax.imshow(grid, cmap="Blues", vmin=0, vmax=1.5, aspect="auto")
for i, row in enumerate(grid):
    for j, wrote in enumerate(row):
        ax.text(j, i, "wrote" if wrote else "-", ha="center", va="center", fontsize=9)
ax.set_xticks(range(len(keys)), keys)
ax.set_yticks(range(len(steps)), [f"step {i + 1}: {n}" for i, n in enumerate(steps)])
ax.set_xlabel("state key"); ax.set_ylabel("super-step")
ax.set_title("Each node returns only the keys it changes (stream_mode='updates')")
plt.tight_layout(); plt.show()
assert all(sum(row) == 1 for row in grid)

## Try it yourself

**1.** Add a `priority` key and a `prioritise` node between `categorise` and `draft_reply`: `"urgent"` if the clean text contains "urgent" or "asap", else `"normal"`. Make `draft_reply` prefix the reply with `[URGENT] ` for urgent tickets.

In [ ]:
# Solution — try it yourself first, then run this
class PriorityTicket(TicketState):
    priority: str

def prioritise(state):
    return {"priority": "urgent" if any(w in state["clean"] for w in ("urgent", "asap")) else "normal"}

def draft_with_priority(state):
    prefix = "[URGENT] " if state["priority"] == "urgent" else ""
    return {"reply": prefix + REPLIES[state["category"]]}

pb = StateGraph(PriorityTicket)
pb.add_sequence([normalise, categorise, prioritise, draft_with_priority])
pb.add_edge(START, "normalise")
prio = pb.compile()
urgent = prio.invoke({"ticket": "URGENT: login error, fix asap"})
print(urgent["priority"], "|", urgent["reply"])
assert urgent["reply"].startswith("[URGENT]") and urgent["category"] == "technical"
assert prio.invoke({"ticket": "hello there"})["priority"] == "normal"

**2.** Write a `keep_max` reducer and use it for a `confidence: float` key. Fan out two nodes from `START` that set confidence 0.7 and 0.9 in the same super-step. The final state should hold 0.9, and there should be no `InvalidUpdateError`.

In [ ]:
# Solution — try it yourself first, then run this
def keep_max(current: float, update: float) -> float:
    return max(current, update)

class Scored(TypedDict):
    confidence: Annotated[float, keep_max]

sg = StateGraph(Scored)
sg.add_node("model_a", lambda s: {"confidence": 0.7})
sg.add_node("model_b", lambda s: {"confidence": 0.9})
sg.add_edge(START, "model_a"); sg.add_edge(START, "model_b")
sg.add_edge("model_a", END); sg.add_edge("model_b", END)
best = sg.compile().invoke({"confidence": 0.0})
print("confidence:", best["confidence"])
assert best["confidence"] == 0.9

**3.** Your manager wants a one-line progress log. Use `stream_mode="updates"` on the audited graph from section 5 to print `normalise -> categorise -> draft_reply`, built from the stream (not hard-coded).

In [ ]:
# Solution — try it yourself first, then run this
order = [node for upd in audited.stream({"ticket": "invoice is wrong", "audit": []}, stream_mode="updates")
         for node in upd]
print(" -> ".join(order))
assert order == ["normalise", "categorise", "draft_reply"]

## Say it in an interview

**30-second answer (LangChain vs LangGraph):** "LangChain is the component layer: model wrappers, prompts, retrievers, tools. LangGraph is the orchestration layer: I model the agent as a state machine with a typed state, nodes that are plain functions returning partial updates, and edges that are fixed or conditional. I compile it into a runnable that can invoke, stream, checkpoint every step and pause for a human. I use LangChain parts inside LangGraph nodes."

**30-second answer (agent vs workflow):** "A workflow is a graph I authored; an agent is a loop where the model picks the next step. In LangGraph both are graphs; the difference is whether the edges are fixed or chosen by a model at run time. Most good production systems are a workflow skeleton with one or two agentic nodes."

**Facts to have ready:**
- A node receives the full state and returns a **partial update**; unchanged keys are left alone.
- Default merge is **overwrite**. Reducers (`Annotated[list, operator.add]`, `add_messages`, custom functions) define how updates combine.
- Execution is in **super-steps**: nodes triggered together run in parallel and their updates are applied at the end of the step. Two parallel writes to a key without a reducer raise `InvalidUpdateError`.
- `add_messages` appends, **replaces by id** and deletes with `RemoveMessage`. That is why chat agents do not use `operator.add`.
- `compile()` validates the wiring (unknown nodes, missing entry point).

**Follow-ups they ask:**
- *"How do you test a graph?"* Nodes are plain functions, so unit-test them with dict inputs; assert the wiring with `get_graph().edges`; run end to end with a fake model.
- *"Why not just write a while loop?"* For one simple agent, a loop is fine and transparent. The graph earns its keep when you need durable pauses, checkpoints, time travel, streaming and parallel branches without writing them yourself.
- *"CrewAI or AutoGen instead?"* Those are conversation-first (agents talk and you hope); LangGraph is graph-first (you draw the process), which is slower to write and much easier to debug.

**Traps:** mutating the state in place (silently lost), misspelt keys (silently dropped), putting large blobs in state (it is saved at every step once you add a checkpointer), and forgetting a reducer on a key that parallel branches write.

## Next

- [02 · Conditional edges and loops](02_conditional_edges_and_loops.ipynb): branches, retries, `Command` and map-reduce with `Send`.
- [03 · Checkpointing, memory and time travel](03_checkpointing_memory_and_time_travel.ipynb): make state survive across calls.
- [LCEL runnables](../13_langchain/02_lcel_runnables_and_output_parsers.ipynb) and [agent loop from scratch](../15_agentic_ai/01_agent_loop_from_scratch.ipynb) for the chain and hand-rolled views of the same ideas.
- Theory: [LangGraph course](../../langgraph/index.html) (chapters 1–3), [LangChain course](../../langchain/index.html), [interview bank](../../ai_interview_prep/index.html).